# 🔬 Harness Engineering — Results Analysis & Interpretation

**Big5Loop Thesis Project** — Systematic evaluation of prompt × model × few-shot configurations for OCEAN personality detection on PANDORA data.

This notebook loads all `harness_*.jsonl` results, computes metrics with bootstrap CIs, generates publication-ready visualizations, and provides thesis-ready interpretation.

In [ ]:
import json
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import spearmanr

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.dpi": 120,
})
warnings.filterwarnings("ignore", category=FutureWarning)

RESULTS_DIR = Path("results")
TRAITS = ["O", "C", "E", "A", "N"]
TRAIT_LABELS = {"O": "Openness", "C": "Conscientiousness", "E": "Extraversion", "A": "Agreeableness", "N": "Neuroticism"}
COLORS = {"O": "#FF6B6B", "C": "#4ECDC4", "E": "#45B7D1", "A": "#96CEB4", "N": "#FFEAA7"}
print("✓ Imports ready")

## 1. Load All Harness Results

In [ ]:
# Load all harness JSONL files
files = sorted(RESULTS_DIR.glob("harness_*.jsonl"))
print(f"Found {len(files)} harness result files:")
for f in files:
    lines = sum(1 for _ in open(f))
    print(f"  {f.name}: {lines} evaluations ({f.stat().st_size/1024:.0f} KB)")

rows = []
for f in files:
    run_id = f.stem.replace("harness_", "")
    with open(f) as fh:
        for line in fh:
            r = json.loads(line)
            r["run_id"] = run_id
            rows.append(r)

df = pd.DataFrame(rows)

# Extract trait columns
for t in TRAITS:
    df[f"gt_{t}"] = df["ground_truth_ocean"].apply(lambda x: float((x or {}).get(t, np.nan)))
    df[f"pr_{t}"] = df["predicted_ocean"].apply(lambda x: float((x or {}).get(t, np.nan)) if x else np.nan)
df["has_prediction"] = df["predicted_ocean"].apply(lambda x: x is not None)

print(f"\n✓ Loaded {len(df)} total evaluations")
print(f"  Models: {sorted(df['model'].unique())}")
print(f"  Prompt sets: {sorted(df['prompt_set'].unique())}")
print(f"  Few-shot counts: {sorted(df['n_shots'].unique())}")
print(f"  Unique samples: {df['sample_id'].nunique()}")
print(f"  Valid predictions: {df['has_prediction'].sum()} ({df['has_prediction'].mean():.0%})")

## 2. Coverage Overview

In [ ]:
# Coverage by model
coverage_model = df.groupby("model")["has_prediction"].agg(["sum", "count", "mean"])
coverage_model.columns = ["Valid", "Total", "Coverage"]
coverage_model = coverage_model.sort_values("Coverage", ascending=False)
print("Coverage by Model:")
print(coverage_model.to_string())
print()

# Coverage by prompt set
coverage_ps = df.groupby("prompt_set")["has_prediction"].agg(["sum", "count", "mean"])
coverage_ps.columns = ["Valid", "Total", "Coverage"]
coverage_ps = coverage_ps.sort_values("Coverage", ascending=False)
print("Coverage by Prompt Set:")
print(coverage_ps.to_string())
print()

# Coverage by few-shot count
coverage_fs = df.groupby("n_shots")["has_prediction"].agg(["sum", "count", "mean"])
coverage_fs.columns = ["Valid", "Total", "Coverage"]
print("Coverage by Few-Shot Count:")
print(coverage_fs.to_string())

## 3. Compute Metrics Per Configuration

In [ ]:
def compute_metrics(group):
    ok = group[group["has_prediction"]]
    n = len(ok)
    total = len(group)
    if n < 3:
        return {"n": n, "total": total, "coverage": n/total if total else 0,
                "macro_r": np.nan, "macro_rho": np.nan, "macro_mae": np.nan,
                "macro_bias": np.nan, "per_trait": {}, "ci_lo": np.nan, "ci_hi": np.nan}
    
    trait_metrics = {}
    pears, maes, biases, spears = [], [], [], []
    for t in TRAITS:
        gt = ok[f"gt_{t}"].dropna().values
        pr = ok[f"pr_{t}"].dropna().values
        min_len = min(len(gt), len(pr))
        if min_len < 3: continue
        gt, pr = gt[:min_len], pr[:min_len]
        
        r = np.corrcoef(gt, pr)[0,1] if np.std(gt) > 0 and np.std(pr) > 0 else 0.0
        rho, _ = spearmanr(gt, pr)
        mae = np.mean(np.abs(pr - gt))
        bias = np.mean(pr - gt)
        
        trait_metrics[t] = {"pearson": r, "spearman": rho if not np.isnan(rho) else 0,
                           "mae": mae, "bias": bias,
                           "gt_mean": np.mean(gt), "gt_std": np.std(gt),
                           "pr_mean": np.mean(pr), "pr_std": np.std(pr)}
        pears.append(r); spears.append(rho if not np.isnan(rho) else 0)
        maes.append(mae); biases.append(bias)
    
    # Bootstrap CI on macro Pearson
    ci_lo, ci_hi = np.nan, np.nan
    if n >= 5:
        rng = np.random.RandomState(42)
        boot_rs = []
        for _ in range(1000):
            idx = rng.choice(n, size=n, replace=True)
            s = ok.iloc[idx]
            rs = []
            for t in TRAITS:
                g, p = s[f"gt_{t}"].values, s[f"pr_{t}"].values
                mask = ~(np.isnan(g)|np.isnan(p))
                if mask.sum() < 3: rs.append(0.0); continue
                rs.append(np.corrcoef(g[mask], p[mask])[0,1] if np.std(g[mask])>0 and np.std(p[mask])>0 else 0)
            boot_rs.append(np.mean(rs))
        ci_lo, ci_hi = np.percentile(boot_rs, 2.5), np.percentile(boot_rs, 97.5)
    
    return {"n": n, "total": total, "coverage": n/total,
            "macro_r": np.mean(pears), "macro_rho": np.mean(spears),
            "macro_mae": np.mean(maes), "macro_bias": np.mean(biases),
            "per_trait": trait_metrics, "ci_lo": ci_lo, "ci_hi": ci_hi}

def composite(m):
    r = m["macro_r"] if not np.isnan(m.get("macro_r", np.nan)) else 0
    return 0.5*r + 0.3*m.get("coverage",0) + 0.2*(1 - (m["macro_mae"] if not np.isnan(m.get("macro_mae",np.nan)) else 1))

# Compute for each (model, prompt_set, n_shots)
configs = {}
for (model, ps, ns), grp in df.groupby(["model", "prompt_set", "n_shots"]):
    configs[(model, ps, ns)] = compute_metrics(grp)

# Build leaderboard
leader = []
for key, m in configs.items():
    leader.append({
        "Model": key[0].split("/")[-1], "Prompt Set": key[1], "Shots": key[2],
        "n": m["n"], "Coverage": m["coverage"],
        "Macro r": m["macro_r"], "95% CI lo": m["ci_lo"], "95% CI hi": m["ci_hi"],
        "Macro ρ": m["macro_rho"], "Macro MAE": m["macro_mae"],
        "Bias": m["macro_bias"], "Composite": composite(m),
    })
leader_df = pd.DataFrame(leader).sort_values("Composite", ascending=False)
print(f"✓ {len(configs)} configurations evaluated, {sum(1 for m in configs.values() if m['n']>=3)} with ≥3 valid")

## 4. 🏆 Leaderboard

In [ ]:
# Show top 15
display_cols = ["Model", "Prompt Set", "Shots", "n", "Coverage", "Macro r", "95% CI lo", "95% CI hi", "Macro MAE", "Composite"]
leader_df[display_cols].head(15).style.format({
    "Coverage": "{:.0%}", "Macro r": "{:.4f}", "95% CI lo": "{:.3f}",
    "95% CI hi": "{:.3f}", "Macro MAE": "{:.4f}", "Composite": "{:.4f}"
}).background_gradient(subset=["Composite"], cmap="YlGn")

## 5. 🤖 Model Comparison

In [ ]:
model_scores = defaultdict(list)
for key, m in configs.items():
    if m["n"] >= 3:
        model_scores[key[0]].append(composite(m))

models = sorted(model_scores, key=lambda x: np.mean(model_scores[x]), reverse=True)
means = [np.mean(model_scores[m]) for m in models]
bests = [max(model_scores[m]) for m in models]
short = [m.split("/")[-1] for m in models]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(models))
w = 0.35
ax.bar(x - w/2, means, w, label="Avg Composite", color="#45B7D1", alpha=0.85)
ax.bar(x + w/2, bests, w, label="Best Composite", color="#FF6B6B", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(short, rotation=30, ha="right")
ax.set_ylabel("Composite Score"); ax.set_title("Model Comparison: Avg vs Best Composite")
ax.legend(); plt.tight_layout(); plt.show()

## 6. 📝 Prompt Set Comparison

In [ ]:
ps_scores = defaultdict(list)
for key, m in configs.items():
    if m["n"] >= 3:
        ps_scores[key[1]].append(composite(m))

ps_list = sorted(ps_scores, key=lambda x: np.mean(ps_scores[x]), reverse=True)
means = [np.mean(ps_scores[p]) for p in ps_list]
bests = [max(ps_scores[p]) for p in ps_list]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(ps_list))
w = 0.35
ax.bar(x - w/2, means, w, label="Avg Composite", color="#4ECDC4", alpha=0.85)
ax.bar(x + w/2, bests, w, label="Best Composite", color="#96CEB4", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(ps_list, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Composite Score"); ax.set_title("Prompt Set Comparison")
ax.legend(); plt.tight_layout(); plt.show()

## 7. 🎯 Few-Shot Ablation

In [ ]:
fs_data = defaultdict(lambda: {"r": [], "cov": [], "comp": []})
for key, m in configs.items():
    if m["n"] >= 3:
        fs_data[key[2]]["r"].append(m["macro_r"])
        fs_data[key[2]]["cov"].append(m["coverage"])
        fs_data[key[2]]["comp"].append(composite(m))

shots = sorted(fs_data)
avg_r = [np.mean(fs_data[s]["r"]) for s in shots]
avg_cov = [np.mean(fs_data[s]["cov"]) for s in shots]
avg_comp = [np.mean(fs_data[s]["comp"]) for s in shots]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(shots, avg_r, "o-", color="#FF6B6B", lw=2, ms=8, label="Macro Pearson r")
ax1.plot(shots, avg_comp, "s-", color="#45B7D1", lw=2, ms=8, label="Composite")
ax1.set_xlabel("# Few-Shot Examples"); ax1.set_ylabel("Score")
ax1.set_title("Few-Shot Ablation: Correlation & Composite")
ax1.legend(); ax1.set_xticks(shots)

ax2.plot(shots, avg_cov, "D-", color="#4ECDC4", lw=2, ms=8)
ax2.set_xlabel("# Few-Shot Examples"); ax2.set_ylabel("Coverage")
ax2.set_title("Few-Shot Ablation: Coverage")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0)); ax2.set_xticks(shots)
plt.tight_layout(); plt.show()

# Table
print("\nFew-Shot Summary:")
for s in shots:
    print(f"  {s:>2}-shot: r={np.mean(fs_data[s]['r']):.4f}, coverage={np.mean(fs_data[s]['cov']):.0%}, composite={np.mean(fs_data[s]['comp']):.4f}")

## 8. 📊 Per-Trait Pearson Heatmap (Top 10 Configs)

In [ ]:
ranked = sorted(
    [(k, m) for k, m in configs.items() if m["n"] >= 3],
    key=lambda x: composite(x[1]), reverse=True
)[:10]

labels, data = [], []
for key, m in ranked:
    labels.append(f"{key[0].split('/')[-1][:12]} / {key[1][:15]} / {key[2]}s")
    data.append([m["per_trait"].get(t, {}).get("pearson", np.nan) for t in TRAITS])

data_arr = np.array(data)
fig, ax = plt.subplots(figsize=(9, max(4, len(labels)*0.55)))
im = ax.imshow(data_arr, cmap="RdYlGn", aspect="auto", vmin=-0.3, vmax=0.6)
ax.set_xticks(range(len(TRAITS)))
ax.set_xticklabels([f"{t}\n({TRAIT_LABELS[t][:4]})" for t in TRAITS])
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=9)

for i in range(len(labels)):
    for j in range(len(TRAITS)):
        v = data_arr[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                   color="white" if abs(v) > 0.3 else "black")

fig.colorbar(im, ax=ax, label="Pearson r", shrink=0.8)
ax.set_title("Per-Trait Pearson Correlation (Top 10 Configs)")
plt.tight_layout(); plt.show()

## 9. Coverage vs Correlation Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
model_colors = {}; cmap = plt.cm.Set2; i = 0

for key, m in configs.items():
    if m["n"] < 2: continue
    model = key[0]
    if model not in model_colors:
        model_colors[model] = cmap(i % 8); i += 1
    ax.scatter(m["coverage"], m["macro_r"], c=[model_colors[model]], s=100, alpha=0.7,
              edgecolors="white", linewidth=0.5)

for model, color in model_colors.items():
    ax.scatter([], [], c=[color], s=100, label=model.split("/")[-1])
ax.legend(loc="upper left"); ax.set_xlabel("Coverage"); ax.set_ylabel("Macro Pearson r")
ax.set_title("Coverage vs Correlation Trade-off (each dot = 1 config)")
ax.axhline(0, color="gray", ls="--", alpha=0.3)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
plt.tight_layout(); plt.show()

## 10. Prediction Bias by Trait

In [ ]:
bias_data = {t: [] for t in TRAITS}
for m in configs.values():
    if m["n"] < 3: continue
    for t in TRAITS:
        b = m["per_trait"].get(t, {}).get("bias")
        if b is not None: bias_data[t].append(b)

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([bias_data[t] for t in TRAITS], tick_labels=[f"{t}\n({TRAIT_LABELS[t][:5]})" for t in TRAITS],
               patch_artist=True)
for patch, t in zip(bp["boxes"], TRAITS):
    patch.set_facecolor(COLORS[t]); patch.set_alpha(0.7)
ax.axhline(0, color="red", ls="--", alpha=0.5)
ax.set_ylabel("Mean Signed Error (Pred - GT)"); ax.set_title("Prediction Bias by Trait")
plt.tight_layout(); plt.show()

print("\nMedian bias per trait:")
for t in TRAITS:
    if bias_data[t]:
        print(f"  {TRAIT_LABELS[t]:20s}: {np.median(bias_data[t]):+.3f}")

## 11. 🏅 Best Configuration — Detailed Analysis

In [ ]:
# Find winner
ranked = sorted([(k, m) for k, m in configs.items() if m["n"] >= 3],
               key=lambda x: composite(x[1]), reverse=True)
wk, wm = ranked[0]

print("=" * 60)
print(f"  🏆 BEST CONFIGURATION")
print("=" * 60)
print(f"  Model:       {wk[0]}")
print(f"  Prompt set:  {wk[1]}")
print(f"  Few-shots:   {wk[2]}")
print(f"  Coverage:    {wm['coverage']:.0%}")
print(f"  Macro r:     {wm['macro_r']:.4f} (95% CI: [{wm['ci_lo']:.3f}, {wm['ci_hi']:.3f}])")
print(f"  Macro ρ:     {wm['macro_rho']:.4f}")
print(f"  Macro MAE:   {wm['macro_mae']:.4f}")
print(f"  Macro Bias:  {wm['macro_bias']:+.4f}")
print(f"  Composite:   {composite(wm):.4f}")
print()

# Per-trait for winner
print(f"  {'Trait':<20} {'Pearson':>8} {'Spearman':>8} {'MAE':>8} {'Bias':>8}")
print(f"  {'─'*20} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
for t in TRAITS:
    tm = wm["per_trait"].get(t, {})
    if tm:
        print(f"  {TRAIT_LABELS[t]:<20} {tm['pearson']:>8.4f} {tm['spearman']:>8.4f} {tm['mae']:>8.4f} {tm['bias']:>+8.4f}")

## 12. Winner: Ground Truth vs Predicted (per trait)

In [ ]:
# Get data for winner config
mask = (df["model"]==wk[0]) & (df["prompt_set"]==wk[1]) & (df["n_shots"]==wk[2]) & df["has_prediction"]
winner_data = df[mask]

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5), sharey=True, sharex=True)
for ax, t in zip(axes, TRAITS):
    gt = winner_data[f"gt_{t}"].values
    pr = winner_data[f"pr_{t}"].values
    valid = ~(np.isnan(gt) | np.isnan(pr))
    ax.scatter(gt[valid], pr[valid], alpha=0.5, s=40, c=COLORS[t], edgecolors="white", lw=0.5)
    ax.plot([-1,1], [-1,1], "k--", alpha=0.3, lw=1)
    r = wm["per_trait"].get(t, {}).get("pearson", 0)
    ax.set_title(f"{t} ({TRAIT_LABELS[t][:4]})\nr={r:.3f}", fontsize=10)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
    if t == "O": ax.set_ylabel("Predicted")
    ax.set_xlabel("Ground Truth")
fig.suptitle(f"Winner: {wk[0].split('/')[-1]} / {wk[1]} / {wk[2]}-shot", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

## 13. 📝 Thesis Interpretation

### Key Findings

Run the cell below to generate a structured interpretation of all results:

In [ ]:
# Model reliability
print("━" * 60)
print("  KEY FINDINGS")
print("━" * 60)

print("\n1️⃣  MODEL RELIABILITY (API Coverage)")
model_cov = defaultdict(list)
for k, m in configs.items(): model_cov[k[0]].append(m["coverage"])
for model in sorted(model_cov, key=lambda x: np.mean(model_cov[x]), reverse=True):
    print(f"   {model.split('/')[-1]:>30}: {np.mean(model_cov[model]):.0%}")

print("\n2️⃣  PROMPT STRATEGY (Avg Macro r, configs with n≥3)")
ps_r = defaultdict(list)
for k, m in configs.items():
    if m["n"] >= 3: ps_r[k[1]].append(m["macro_r"])
for ps in sorted(ps_r, key=lambda x: np.mean(ps_r[x]), reverse=True):
    print(f"   {ps:>30}: r = {np.mean(ps_r[ps]):.4f}")

print("\n3️⃣  TRAIT DIFFICULTY (avg r across configs)")
trait_r = defaultdict(list)
for m in configs.values():
    if m["n"] >= 3:
        for t in TRAITS:
            if t in m["per_trait"]: trait_r[t].append(m["per_trait"][t]["pearson"])
for t in sorted(trait_r, key=lambda x: np.mean(trait_r[x]), reverse=True):
    print(f"   {TRAIT_LABELS[t]:20s} ({t}): r = {np.mean(trait_r[t]):.4f}")

print("\n4️⃣  SYSTEMATIC BIASES")
trait_bias = defaultdict(list)
for m in configs.values():
    if m["n"] >= 3:
        for t in TRAITS:
            if t in m["per_trait"]: trait_bias[t].append(m["per_trait"][t]["bias"])
for t in TRAITS:
    if trait_bias[t]:
        b = np.mean(trait_bias[t])
        d = "over-predicts ⬆" if b > 0.05 else "under-predicts ⬇" if b < -0.05 else "well-calibrated ✓"
        print(f"   {TRAIT_LABELS[t]:20s}: bias = {b:+.3f} ({d})")

print("\n━" * 60)
print("  THESIS RECOMMENDATION")
print("━" * 60)
wk, wm = ranked[0]
print(f"\n  ✅ DEPLOY: {wk[0]}")
print(f"     Prompt: {wk[1]} ({wk[2]}-shot)")
print(f"     Composite: {composite(wm):.4f}")
print(f"     Macro r: {wm['macro_r']:.4f}, Coverage: {wm['coverage']:.0%}")

if wm["macro_r"] < 0.2:
    print("\n  ⚠️  Macro Pearson < 0.20 — weak individual-level correlation.")
    print("     Expected for single-message personality detection.")
    print("     Frame as: 'consistent with psychometric limits of short-text")
    print("     personality assessment (Schwartz et al., 2013; Gjurković et al., 2021)'")
elif wm["macro_r"] < 0.3:
    print("\n  ℹ️  Moderate correlation — competitive with PANDORA baselines.")
else:
    print("\n  🎯 Strong correlation — exceeds typical baselines for this task.")
print()

## 14. Export Summary

Save the leaderboard and metrics to CSV for further analysis:

In [ ]:
# Save leaderboard
leader_df.to_csv(RESULTS_DIR / "harness_leaderboard.csv", index=False)
print(f"✓ Saved: {RESULTS_DIR / 'harness_leaderboard.csv'}")

# Save per-trait metrics for winner
trait_rows = []
for t in TRAITS:
    tm = wm["per_trait"].get(t, {})
    if tm: trait_rows.append({"trait": t, "label": TRAIT_LABELS[t], **tm})
trait_df = pd.DataFrame(trait_rows)
trait_df.to_csv(RESULTS_DIR / "winner_trait_metrics.csv", index=False)
print(f"✓ Saved: {RESULTS_DIR / 'winner_trait_metrics.csv'}")

print(f"\n✅ Analysis complete! Re-run after your big benchmark finishes for updated results.")